# Decision Tree with Scikit-Learn - Day 7

## Variable Importances (Feature Importance)

Variable importance (also known as feature importance) is a score that indicates how "important" a feature is to the model.

**Example:** For a model with two input features "f1" and "f2", if variable importances are {f1=5.8, f2=2.5}, then feature "f1" is more important than feature "f2".

Variable importance is a simple way to understand how a decision tree works.

---

## Types of Variable Importances

### Model Agnostic Importances
- Permutation variable importances (can be applied to any model)

### Decision Tree Specific Importances

| Type | Description | Scikit-Learn |
|------|-------------|-------------|
| Gini Importance | Total decrease in node impurity | `feature_importances_` |
| Permutation Importance | Decrease in score when feature is shuffled | `permutation_importance` |

---

## What Variable Importances Tell Us

| Aspect | Information Provided |
|--------|---------------------|
| Model | Which features the model relies on |
| Dataset | Which features contain predictive signal |
| Training process | How the algorithm utilized each feature |

---

## Interpreting Different Importances

**High number of nodes + Low permutation importance:**
- Feature might be hard to generalize
- Could hurt model quality
- Model trying but failing to learn pattern

**High importance across all measures:**
- Feature is likely genuinely important
- Strong signal in the data

**Best practice:** Look at several variable importances simultaneously for complete picture

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

In [ ]:
# Create data where only first feature matters
np.random.seed(42)
n_samples = 100
n_features = 5

X = np.random.rand(n_samples, n_features)
# Only first feature affects target
y = 5 * X[:, 0] + np.random.randn(n_samples) * 0.5

print("Data created:")
print(f"  Samples: {n_samples}")
print(f"  Features: {n_features}")
print(f"  Only Feature 0 affects the target")

In [ ]:
# Train Decision Tree
model = DecisionTreeRegressor(max_depth=5)
model.fit(X, y)

print(f"Train R2 Score: {model.score(X, y):.4f}")

In [ ]:
# Method 1: Built-in Feature Importances (Gini Importance)
print("\n" + "="*50)
print("Built-in Feature Importances (Gini Importance)")
print("="*50)

for i, imp in enumerate(model.feature_importances_):
    print(f"Feature {i}: {imp:.4f}")

print(f"\nTotal: {np.sum(model.feature_importances_):.4f}")

In [ ]:
# Method 2: Permutation Importance
print("\n" + "="*50)
print("Permutation Importance")
print("="*50)

result = permutation_importance(model, X, y, n_repeats=10, random_state=42)

for i, imp in enumerate(result.importances_mean):
    std = result.importances_std[i]
    print(f"Feature {i}: {imp:.4f} (+/- {std:.4f})")

print(f"\nHigher score = more important")

In [ ]:
# Visualize both importance methods
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Built-in importance
axes[0].bar(range(n_features), model.feature_importances_, color='steelblue')
axes[0].set_xlabel('Feature Index')
axes[0].set_ylabel('Gini Importance')
axes[0].set_title('Built-in Feature Importance')
axes[0].set_xticks(range(n_features))
axes[0].grid(alpha=0.3)

# Permutation importance
axes[1].bar(range(n_features), result.importances_mean, yerr=result.importances_std, 
            color='coral', capsize=5)
axes[1].set_xlabel('Feature Index')
axes[1].set_ylabel('Permutation Importance')
axes[1].set_title('Permutation Importance')
axes[1].set_xticks(range(n_features))
axes[1].grid(alpha=0.3)

plt.suptitle('Feature Importance Comparison', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze tree structure
print("\n" + "="*50)
print("Tree Analysis")
print("="*50)

# Count how many times each feature is used
feature_usage = np.zeros(n_features)
for node in range(model.tree_.node_count):
    if model.tree_.feature[node] >= 0:
        feature_usage[model.tree_.feature[node]] += 1

for i, usage in enumerate(feature_usage):
    print(f"Feature {i} used in {int(usage)} nodes")

print(f"\nTotal nodes: {model.tree_.node_count}")
print(f"Total leaves: {model.tree_.n_leaves}")

In [ ]:
# Permutation importance from scratch
def permutation_importance_scratch(model, X, y):
    baseline = model.score(X, y)
    importances = []
    
    for col in range(X.shape[1]):
        X_permuted = X.copy()
        np.random.shuffle(X_permuted[:, col])
        perm_score = model.score(X_permuted, y)
        importance = baseline - perm_score
        importances.append(importance)
    
    return importances

print("\n" + "="*50)
print("Permutation Importance (From Scratch)")
print("="*50)

imp_scratch = permutation_importance_scratch(model, X, y)
for i, imp in enumerate(imp_scratch):
    print(f"Feature {i}: {imp:.4f}")

print("\nLarger decrease = more important feature")

In [ ]:
# Create feature importance summary table
print("\n" + "="*60)
print("FEATURE IMPORTANCE SUMMARY")
print("="*60)

print(f"{'Feature':<10} {'Gini':<12} {'Permutation':<12} {'Node Count':<12} {'Important?':<12}")
print("-"*60)

for i in range(n_features):
    gini = model.feature_importances_[i]
    perm = result.importances_mean[i]
    nodes = feature_usage[i]
    
    if i == 0:
        important = "YES (True signal)"
    elif gini > 0.1:
        important = "Maybe"
    else:
        important = "No"
    
    print(f"Feature {i:<8} {gini:<12.4f} {perm:<12.4f} {int(nodes):<12} {important:<12}")

print("="*60)

In [ ]:
# Day 7 Completed
print("\n" + "="*50)
print("DAY 7 COMPLETED")
print("="*50)
print("Topics covered:")
print("- Variable/Feature Importance concepts")
print("- Built-in Gini importance in Scikit-Learn")
print("- Permutation importance from sklearn.inspection")
print("- Permutation importance from scratch")
print("- Interpreting different importance measures")
print("- Feature usage analysis in tree nodes")
print("="*50)"
print("\nNext: Day 8 - Random Forest")